# 2024 311 Encampment Reports -> SatScan prep (public dataset version)

Source: `311_Encampment_Reports_2C_2024_2C_City_of_San_Diego.csv`

Differences from the original pipeline:
- No `Camp` flag / hierarchy columns -> every row is already an encampment report.
- No shapely `geometry` -> `location_id` is derived from rounded `lat`/`lng`.
- `date` comes from `date_requested` (UTC timestamp -> date).

In [1]:
import pandas as pd

CSV_PATH = "311_Encampment_Reports%2C_2024%2C_City_of_San_Diego.csv"
df = pd.read_csv(CSV_PATH)
df.shape

(59059, 26)

In [2]:
# Equivalent of `df[df["Camp"] == True]`.
# This export is already filtered to encampment reports; keep an explicit filter
# so the intent matches the original and to guard against future exports.
camp_data = df[df["service_name"].str.strip().str.lower() == "encampment"].copy()
camp_data["service_name"].value_counts()

Encampment    59059
Name: service_name, dtype: int64

In [3]:
# Keep only what the .cas/.geo files need (replaces the cols_to_drop step)
cols_to_keep = ["service_request_id", "date_requested", "lat", "lng"]
camp_data = camp_data[cols_to_keep]
camp_data.head()

,service_request_id,date_requested,lat,lng
0,4563179,2024/01/01 08:32:00+00,32.716936,-117.154526
1,4563180,2024/01/01 08:36:00+00,32.748568,-117.131777
2,4563181,2024/01/01 08:38:00+00,32.916662,-117.116820
3,4563187,2024/01/01 09:14:00+00,32.713853,-117.158849
4,4563192,2024/01/01 11:23:00+00,32.705174,-117.148147


In [5]:
# `date` column, normalized to day resolution
camp_data["date"] = pd.to_datetime(camp_data["date_requested"], utc=True).dt.date
camp_data["date"] = pd.to_datetime(camp_data["date"])
camp_data = camp_data.drop(columns=["date_requested"])
camp_data.head()

,service_request_id,lat,lng,date
0,4563179,32.716936,-117.154526,2024-01-01
1,4563180,32.748568,-117.131777,2024-01-01
2,4563181,32.916662,-117.116820,2024-01-01
3,4563187,32.713853,-117.158849,2024-01-01
4,4563192,32.705174,-117.148147,2024-01-01


In [6]:
# Drop records without coordinates (replaces the geometry isna step)
print("missing coords:", camp_data[["lat", "lng"]].isna().any(axis=1).sum())
camp_data = camp_data.dropna(subset=["lat", "lng"])

# Guard against 0,0 / null-island placeholders
camp_data = camp_data[(camp_data["lat"] != 0) & (camp_data["lng"] != 0)]
camp_data.shape

missing coords: 17


(59042, 4)

In [7]:
# Consistent location_id for identical coordinates.
# COORD_PRECISION=6 mirrors exact geometry matching; 5 (~1m) or 4 (~11m) will
# snap near-identical reports onto one location, which SaTScan generally prefers.
COORD_PRECISION = 6

camp_data["latitude"] = camp_data["lat"].round(COORD_PRECISION)
camp_data["longitude"] = camp_data["lng"].round(COORD_PRECISION)
camp_data = camp_data.drop(columns=["lat", "lng"])

unique_coords = (
    camp_data[["latitude", "longitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)
unique_coords["location_id"] = range(1, len(unique_coords) + 1)

camp_data = camp_data.merge(unique_coords, on=["latitude", "longitude"], how="left")
print("unique locations:", len(unique_coords))
camp_data.head()

unique locations: 56470


,service_request_id,date,latitude,longitude,location_id
0,4563179,2024-01-01,32.716936,-117.154526,19066
1,4563180,2024-01-01,32.748568,-117.131777,32970
2,4563181,2024-01-01,32.916662,-117.116820,55434
3,4563187,2024-01-01,32.713853,-117.158849,13031
4,4563192,2024-01-01,32.705174,-117.148147,2812


In [8]:
# Full-duplicate removal.
# NOTE: service_request_id is unique per report, so including it means nothing is
# dropped. Dedupe on the fields that actually define a duplicate observation.
dupes = camp_data[camp_data.duplicated(subset=["date", "location_id"], keep=False)]
print("rows sharing a date+location:", dupes.shape[0])

# The original pipeline collapsed these. Set to True to reproduce that behavior;
# leave False to let repeat reports at the same spot count as multiple cases.
COLLAPSE_SAME_DAY_SAME_LOCATION = False
if COLLAPSE_SAME_DAY_SAME_LOCATION:
    camp_data = camp_data.drop_duplicates(subset=["date", "location_id"], keep="first")
camp_data.shape

rows sharing a date+location: 964


(59042, 5)

## Create cas file

In [9]:
aggregated_df = camp_data.groupby(["date", "location_id"]).size().reset_index(name="cases")
aggregated_df = aggregated_df.sort_values(by=["date", "location_id"])
aggregated_df["case_id"] = range(1, len(aggregated_df) + 1)

cas_df = aggregated_df[["case_id", "date", "cases", "location_id"]].copy()
cas_df["date"] = cas_df["date"].dt.strftime("%Y/%m/%d")
cas_df.head()

,case_id,date,cases,location_id
0,1,2024/01/01,1,1084
1,2,2024/01/01,1,2397
2,3,2024/01/01,1,2649
3,4,2024/01/01,1,2812
4,5,2024/01/01,1,2820


In [10]:
print(cas_df.shape)
print("total cases:", cas_df["cases"].sum())
cas_df[cas_df["cases"] > 1].head()

(58500, 4)
total cases: 59042


,case_id,date,cases,location_id
568,569,2024/01/03,2,49998
609,610,2024/01/04,2,2193
827,828,2024/01/04,2,43146
920,921,2024/01/05,2,1821
933,934,2024/01/05,2,4480


In [11]:
import os

OUT_DIR = "satscan_usable"
os.makedirs(OUT_DIR, exist_ok=True)

cas_file_path = os.path.join(OUT_DIR, "2024_camps.cas")
cas_df.to_csv(cas_file_path, index=False, header=True)
print("wrote", cas_file_path)

wrote satscan_usable/2024_camps.cas


## Create geo file

In [12]:
geo_df = camp_data[["location_id", "latitude", "longitude"]].drop_duplicates(subset=["location_id"])

if not ((geo_df["latitude"].between(-90, 90)) & (geo_df["longitude"].between(-180, 180))).all():
    raise ValueError("Invalid coordinates found in the data.")

geo_df = geo_df.sort_values(by="location_id")
print(geo_df.shape)
geo_df.head()

(56470, 3)


,location_id,latitude,longitude
55899,1,32.543377,-117.034032
15248,2,32.543608,-117.050214
30357,3,32.543609,-117.051023
33376,4,32.543622,-117.049536
6628,5,32.543627,-117.049900


In [13]:
geo_file_path = os.path.join(OUT_DIR, "2024_camps.geo")
geo_df.to_csv(geo_file_path, index=False, header=True)
print("wrote", geo_file_path)

wrote satscan_usable/2024_camps.geo


## Validation

In [14]:
# 1:1 location_id <-> coordinate pair
assert not geo_df.duplicated(subset=["latitude", "longitude"]).any(), "duplicate coord pairs"
assert not geo_df.duplicated(subset=["location_id"]).any(), "duplicate location_ids"

# Every location in .cas exists in .geo
assert set(cas_df["location_id"]).issubset(set(geo_df["location_id"])), "cas has locations missing from geo"

# Case counts round-trip
check = camp_data.groupby(["date", "location_id"]).size().reset_index(name="camp_data_count")
check["date"] = check["date"].dt.strftime("%Y/%m/%d")
merged_df = check.merge(cas_df, on=["date", "location_id"], how="inner")
merged_df["match"] = merged_df["camp_data_count"] == merged_df["cases"]
assert len(merged_df) == len(cas_df) and merged_df["match"].all(), "count mismatch"
print("All counts match!")

All counts match!


In [15]:
from datetime import datetime

min_date = cas_df["date"].min()
max_date = cas_df["date"].max()
print(f"Study Period Start: {min_date}")
print(f"Study Period End: {max_date}")

start_date = datetime.strptime(min_date, "%Y/%m/%d")
end_date = datetime.strptime(max_date, "%Y/%m/%d")
print(f"Total days in study period: {(end_date - start_date).days + 1}")

Study Period Start: 2024/01/01
Study Period End: 2024/12/31
Total days in study period: 366
